# Sydney candidates vs pipeline output — rejection reasons

Loads all LTV pipeline output parquets, matches them to Sydney LTV candidates (from `input/SydneyLTVs.csv`), and reports **filter_reason** for each Sydney candidate (passed vs. which filter removed them).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from astropy.coordinates import SkyCoord
from astropy import units as u

# Paths (resolved by walking upward to the repo root)
cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (cwd, *cwd.parents) if (p / "input" / "SydneyLTVs.csv").exists()), cwd)
LTV_OUTPUT_DIR = REPO_ROOT / "output" / "ltv"
SYDNEY_CSV = REPO_ROOT / "input" / "SydneyLTVs.csv"

## 1. Discover pipeline output parquets and load Sydney list

In [6]:
# Pipeline outputs: LTvar12-12.5_pipeline.parquet, etc.
parquet_files = sorted(LTV_OUTPUT_DIR.glob("*_pipeline.parquet"))
if not parquet_files:
    parquet_files = sorted(LTV_OUTPUT_DIR.glob("*_pipeline.csv"))
print(f"Found {len(parquet_files)} pipeline output(s): {[p.name for p in parquet_files]}")

Found 0 pipeline output(s): []


In [7]:
def load_sydney(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    # Normalize 2MASS for matching (same as crossmatch_sydney_ltv)
    df["2MASS_clean"] = (
        df["2MASS"].astype(str)
        .str.replace(r"2MJ", "", regex=False)
        .str.replace(r"\$\\+\$", "+", regex=True)
        .str.replace(r"\$\\-\$", "-", regex=True)
        .str.strip()
    )
    return df

def parse_2mass_ra_dec(s: str):
    """Parse Sydney-style 2MASS ID to (ra_deg, dec_deg). Same logic as crossmatch._sydney_2mass_to_ra_dec.
    Format JHHMMSSs±DDMMSSs: RA 8 chars, Dec sign + 7 chars (dd mm ss.s)."""
    s = (str(s) or "").strip().replace("2MJ", "").replace("$+$", "+").replace("$-", "-")
    if len(s) < 15 or s[8:9] not in ("+", "-"):
        return None, None
    try:
        ra_str = s[:8]
        dec_str = s[8:]
        ra_h = int(ra_str[0:2]) + int(ra_str[2:4])/60.0 + (int(ra_str[4:6]) + int(ra_str[6:8])/100.0)/3600.0
        ra_deg = 15.0 * ra_h
        sign = 1 if dec_str[0] == "+" else -1
        dec_d = int(dec_str[1:3]) + int(dec_str[3:5])/60.0 + (int(dec_str[5:7]) + int(dec_str[7:8])/10.0)/3600.0
        dec_deg = sign * dec_d
        return ra_deg, dec_deg
    except (ValueError, IndexError):
        return None, None

if SYDNEY_CSV.exists():
    sydney = load_sydney(SYDNEY_CSV)
    ra_dec = [parse_2mass_ra_dec(x) for x in sydney["2MASS"].values]
    sydney["ra_deg"] = [r[0] for r in ra_dec]
    sydney["dec_deg"] = [r[1] for r in ra_dec]
    sydney_valid = sydney[sydney["ra_deg"].notna()].copy()
    print(f"Sydney candidates: {len(sydney)} total, {len(sydney_valid)} with parseable coords")
else:
    sydney = pd.DataFrame()
    sydney_valid = pd.DataFrame()
    print(f"Sydney file not found: {SYDNEY_CSV}")

AttributeError: 'str' object has no attribute 'exists'

### Why so many "NOT found"? Pipeline output is per magnitude bin

In [8]:
# Pipeline parquets cover specific g-band bins (e.g. 12_12.5, 12.5_13, ...).
# A Sydney candidate only appears in a parquet if it was in that bin's INPUT (from ltv.core).
# So "412 not found" usually means: their g mag falls in bins you didn't run (e.g. 13.5_14, 14_14.5, 14.5_15).
if not sydney_valid.empty and parquet_files:
    bin_lo_hi = []
    for p in parquet_files:
        stem = p.stem.replace("_pipeline", "").replace("LTvar", "")
        parts = stem.replace("-", "_").split("_")
        if len(parts) == 2:
            bin_lo_hi.append((float(parts[0]), float(parts[1])))
    print("Pipeline mag bins (g range) you have:", bin_lo_hi)
    if "g" in sydney_valid.columns:
        g = pd.to_numeric(sydney_valid["g"], errors="coerce")
        g_valid = g.dropna()
        print(f"Sydney g (parseable): min={g_valid.min():.2f}, max={g_valid.max():.2f}, median={g_valid.median():.2f}")
        for (lo, hi) in bin_lo_hi:
            in_bin = (g_valid >= lo) & (g_valid < hi)
            print(f"  Sydney in [{lo}, {hi}): {in_bin.sum()}")
        other = g_valid[(g_valid < bin_lo_hi[0][0]) | (g_valid >= bin_lo_hi[-1][1])] if bin_lo_hi else g_valid
        if len(other) > 0:
            print(f"  Sydney outside your bins (g < {bin_lo_hi[0][0]} or g >= {bin_lo_hi[-1][1]}): {len(other)}")
else:
    print("No Sydney or parquet files to compare.")

No Sydney or parquet files to compare.


## 2. Match pipeline candidates to Sydney (by ASAS-SN ID, 2MASS, or coordinates)

In [16]:
def match_sydney_to_pipeline(sydney_df: pd.DataFrame, pipe_df: pd.DataFrame, coord_arcsec: float = 3.0):
    """
    For each Sydney row, find matching row(s) in pipeline output.
    Returns a DataFrame with one row per (Sydney candidate, mag_bin) with columns from both + filter_reason.
    """
    if pipe_df.empty or sydney_df.empty:
        return pd.DataFrame()

    # Prefer ASAS-SN ID if both have it
    asas_col = "ASAS-SN ID"
    id_matches = []
    if asas_col in sydney_df.columns and asas_col in pipe_df.columns:
        sydney_ids = set(sydney_df[asas_col].dropna().astype(str).str.strip())
        pipe_with_id = pipe_df[pipe_df[asas_col].notna()].copy()
        pipe_with_id["_id"] = pipe_with_id[asas_col].astype(str).str.strip()
        for sid in sydney_ids:
            hit = pipe_with_id[pipe_with_id["_id"] == sid]
            if not hit.empty:
                id_matches.append((sid, hit.index[0], "asas_sn_id"))
    
    # 2MASS match if pipeline has 2MASS_ID
    pipe_2mass = None
    if "2MASS_ID" in pipe_df.columns and "2MASS_clean" in sydney_df.columns:
        pipe_2mass = pipe_df[pipe_df["2MASS_ID"].notna()].copy()
        pipe_2mass["2MASS_clean"] = pipe_2mass["2MASS_ID"].astype(str).str.strip()

    # Coordinate match for Sydney rows with ra_deg/dec_deg
    coord_cols = ["ra_deg", "dec_deg"]
    use_coord = (
        "ra_deg" in pipe_df.columns
        and "dec_deg" in pipe_df.columns
        and "ra_deg" in sydney_df.columns
        and "dec_deg" in sydney_df.columns
    )
    matched_sydney_idx = set()  # Sydney index already matched by ID
    matched_pipe_idx = set()   # Pipeline index already matched
    for _, row in id_matches:
        matched_pipe_idx.add(row)

    rows_by_sydney = {}  # sydney_idx -> list of (pipe_idx, match_type)
    for i, srow in sydney_df.iterrows():
        rows_by_sydney[i] = []

    # Apply ID matches
    for sid, pipe_idx, _ in id_matches:
        sydney_row = sydney_df[sydney_df[asas_col].astype(str).str.strip() == sid].iloc[0]
        si = sydney_df[sydney_df[asas_col].astype(str).str.strip() == sid].index[0]
        if si not in rows_by_sydney:
            rows_by_sydney[si] = []
        rows_by_sydney[si].append((pipe_idx, "asas_sn_id"))
        matched_sydney_idx.add(si)
        matched_pipe_idx.add(pipe_idx)

    # 2MASS matches for Sydney not yet matched
    if pipe_2mass is not None:
        for si, srow in sydney_df.iterrows():
            if si in matched_sydney_idx:
                continue
            clean = srow.get("2MASS_clean")
            if pd.isna(clean):
                continue
            hit = pipe_2mass[pipe_2mass["2MASS_clean"] == str(clean).strip()]
            if not hit.empty:
                pipe_idx = hit.index[0]
                if pipe_idx not in matched_pipe_idx:
                    rows_by_sydney[si].append((pipe_idx, "2mass_id"))
                    matched_pipe_idx.add(pipe_idx)
                matched_sydney_idx.add(si)

    # Coordinate match for remaining Sydney with valid coords
    if use_coord:
        sydney_coord = sydney_df["ra_deg"].notna() & sydney_df["dec_deg"].notna()
        s_coords = SkyCoord(
            ra=sydney_df.loc[sydney_coord, "ra_deg"].values * u.deg,
            dec=sydney_df.loc[sydney_coord, "dec_deg"].values * u.deg,
        )
        d_coords = SkyCoord(
            ra=pipe_df["ra_deg"].values * u.deg,
            dec=pipe_df["dec_deg"].values * u.deg,
        )
        idx_s, sep, _ = s_coords.match_to_catalog_sky(d_coords)
        for k, (si, sep_arcsec) in enumerate(zip(sydney_df.loc[sydney_coord].index, sep.arcsec)):
            if sep_arcsec > coord_arcsec:
                continue
            pipe_idx = pipe_df.index[int(idx_s[k])]
            if si in matched_sydney_idx and rows_by_sydney[si]:
                continue  # already matched by ID
            rows_by_sydney[si].append((pipe_idx, "coord"))
            matched_pipe_idx.add(pipe_idx)

    # Build result: one row per (Sydney, pipeline match) with filter_reason from pipeline
    result_rows = []
    for si, pipe_indices in rows_by_sydney.items():
        srow = sydney_df.loc[si]
        for pipe_idx, match_type in pipe_indices:
            prow = pipe_df.loc[pipe_idx]
            reason = prow.get("filter_reason", pd.NA)
            result_rows.append({
                **srow.to_dict(),
                "_pipe_idx": pipe_idx,
                "_match_type": match_type,
                "filter_reason": reason,
            })
    if not result_rows:
        return pd.DataFrame()
    out = pd.DataFrame(result_rows)
    if "_pipe_idx" in out.columns:
        out = out.drop(columns=["_pipe_idx"])
    return out

In [17]:
# Simpler: for each parquet, match Sydney (by coord) and attach filter_reason
def match_sydney_to_pipeline_simple(sydney_valid: pd.DataFrame, pipe_df: pd.DataFrame, mag_bin: str, coord_arcsec: float = 10.0):
    """Match Sydney candidates to pipeline output by coordinates; return table with filter_reason and mag_bin."""
    if pipe_df.empty or sydney_valid.empty:
        return pd.DataFrame()
    # Pipeline may use ra_deg/dec_deg or ra/dec
    pipe_ra = pipe_df["ra_deg"] if "ra_deg" in pipe_df.columns else pipe_df["ra"]
    pipe_dec = pipe_df["dec_deg"] if "dec_deg" in pipe_df.columns else pipe_df["dec"]
    if pipe_ra.isna().all() or pipe_dec.isna().all() or not all(c in sydney_valid.columns for c in ["ra_deg", "dec_deg"]):
        return pd.DataFrame()
    s_coords = SkyCoord(
        ra=sydney_valid["ra_deg"].values * u.deg,
        dec=sydney_valid["dec_deg"].values * u.deg,
    )
    d_coords = SkyCoord(
        ra=pipe_ra.values * u.deg,
        dec=pipe_dec.values * u.deg,
    )
    idx_s, sep, _ = s_coords.match_to_catalog_sky(d_coords)
    sep_arcsec = sep.arcsec
    matched = sep_arcsec < coord_arcsec
    if not matched.any():
        return pd.DataFrame()
    # One row per Sydney match: Sydney info + filter_reason from pipeline + mag_bin
    sydney_matched = sydney_valid[matched].copy()
    sydney_matched = sydney_matched.reset_index(drop=True)
    pipe_idx_matched = np.where(matched)[0]
    pipe_loc = idx_s[matched]  # pipeline row index for each matched Sydney
    sydney_matched["mag_bin"] = mag_bin
    sydney_matched["filter_reason"] = (
        pipe_df["filter_reason"].iloc[pipe_loc].values
        if "filter_reason" in pipe_df.columns
        else pd.NA
    )
    sydney_matched["sep_arcsec"] = sep_arcsec[matched]
    return sydney_matched

## 3. Load each pipeline parquet and collect Sydney matches with rejection reasons

In [18]:
all_matches = []  # list of DataFrames, one per parquet
mag_bins = []

for path in parquet_files:
    # Infer mag bin from filename, e.g. LTvar12-12.5_pipeline.parquet -> 12_12.5
    stem = path.stem.replace("_pipeline", "")
    mag_bin = stem.replace("LTvar", "").replace("-", "_")  # 12_12.5
    mag_bins.append(mag_bin)
    df = pd.read_parquet(path) if path.suffix.lower() in (".parquet", ".pq") else pd.read_csv(path)
    if "filter_reason" not in df.columns:
        df["filter_reason"] = pd.NA
    matched = match_sydney_to_pipeline_simple(sydney_valid, df, mag_bin=mag_bin, coord_arcsec=10.0)
    if not matched.empty:
        all_matches.append(matched)

if all_matches:
    sydney_reasons = pd.concat(all_matches, ignore_index=True)
    # Dedupe: same Sydney source can match in multiple bins; keep one row per (2MASS, mag_bin) or aggregate
    print(f"Total Sydney–pipeline matches (across bins): {len(sydney_reasons)}")
else:
    sydney_reasons = pd.DataFrame()
    print("No pipeline parquets or no Sydney matches.")

Total Sydney–pipeline matches (across bins): 4


## 4. Per-mag-bin recovery fraction

In [19]:
# Recovery = of Sydney candidates with g in this bin, how many appear in this bin's pipeline output.
# So with a subset of parquets you get a faithful recovery rate PER BIN (not diluted by bins you didn't run).
def _sydney_key(row):
    if "2MASS_clean" in row.index and pd.notna(row.get("2MASS_clean")):
        return ("2MASS", str(row["2MASS_clean"]).strip())
    return ("coord", (round(row["ra_deg"], 6), round(row["dec_deg"], 6)))

recovery_rows = []
for path in parquet_files:
    stem = path.stem.replace("_pipeline", "").replace("LTvar", "")
    parts = stem.replace("-", "_").split("_")
    if len(parts) != 2:
        continue
    mag_bin = stem.replace("-", "_")
    g_lo, g_hi = float(parts[0]), float(parts[1])
    # Sydney with parseable coords whose g falls in this bin
    if sydney_valid.empty or "g" not in sydney_valid.columns:
        n_sydney = 0
    else:
        g = pd.to_numeric(sydney_valid["g"], errors="coerce")
        in_bin = (g >= g_lo) & (g < g_hi)
        sydney_in_bin = sydney_valid.loc[in_bin]
        n_sydney = len(sydney_in_bin)
    # Matched in this bin (from our coord match to this parquet)
    if sydney_reasons.empty or "mag_bin" not in sydney_reasons.columns:
        matched_keys = set()
    else:
        in_bin_df = sydney_reasons[sydney_reasons["mag_bin"] == mag_bin]
        matched_keys = set(_sydney_key(r) for _, r in in_bin_df.iterrows())
    # Recovered = Sydney in this bin that are in matched_keys
    if n_sydney == 0:
        n_recovered = 0
        recovery_frac = np.nan
    else:
        sydney_keys_bin = set(_sydney_key(r) for _, r in sydney_in_bin.iterrows())
        n_recovered = len(sydney_keys_bin & matched_keys)
        recovery_frac = n_recovered / n_sydney
    recovery_rows.append({
        "mag_bin": mag_bin,
        "g_lo": g_lo,
        "g_hi": g_hi,
        "N_sydney_in_bin": n_sydney,
        "N_recovered": n_recovered,
        "recovery_fraction": recovery_frac,
    })

recovery_df = pd.DataFrame(recovery_rows)
if not recovery_df.empty:
    print("Sydney candidates within each mag bin g-range (from Sydney CSV):")
    for _, r in recovery_df.iterrows():
        print(f"  {r['mag_bin']}: g in [{r['g_lo']}, {r['g_hi']}) -> {int(r['N_sydney_in_bin'])} Sydney candidates")
    print("Per-mag-bin recovery (of those, how many appear in pipeline output for that bin):")
    display(recovery_df)
    if "g" not in sydney_valid.columns:
        print("Note: Sydney has no 'g' column; N_sydney_in_bin is 0 and recovery_fraction is NaN.")
else:
    print("No parquet files or no mag bins to report.")

Sydney candidates in CSV: 1208 total, 416 with parseable coords
Per-mag-bin recovery (Sydney candidates with g in [g_lo, g_hi) that appear in pipeline output for that bin):


,mag_bin,g_lo,g_hi,N_sydney_in_bin,N_recovered,recovery_fraction
0,12_12.5,12.0,12.5,0,0,NaN
1,12.5_13,12.5,13.0,0,0,NaN
2,13_13.5,13.0,13.5,75,4,0.053333


## 5. Summary: rejection reasons for Sydney candidates

In [20]:
if not sydney_reasons.empty:
    print("--- Counts by filter_reason (Sydney candidates only) ---")
    print(sydney_reasons["filter_reason"].value_counts(dropna=False).to_string())
    print("\n--- By mag_bin ---")
    print(sydney_reasons.groupby("mag_bin")["filter_reason"].value_counts().unstack(fill_value=0).to_string())

--- Counts by filter_reason (Sydney candidates only) ---
filter_reason
neighbor_high_pm    2
passed              2

--- By mag_bin ---
filter_reason  neighbor_high_pm  passed
mag_bin                                
13_13.5                       2       2


In [21]:
if not sydney_reasons.empty:
    # Table: Sydney 2MASS, Class, mag_bin, filter_reason
    cols = [c for c in ["2MASS", "2MASS_clean", "ASAS-SN ID", "Class", "g", "mag_bin", "filter_reason", "sep_arcsec"] if c in sydney_reasons.columns]
    display(sydney_reasons[cols].sort_values(["filter_reason", "mag_bin"]))

,2MASS,2MASS_clean,ASAS-SN ID,Class,g,mag_bin,filter_reason,sep_arcsec
0,2MJ01564514$+$5918432,01564514$+$5918432,NaN,AGB,13.310,13_13.5,neighbor_high_pm,0.151392
1,2MJ18393017$+$0752032,18393017$+$0752032,NaN,AGB,13.417,13_13.5,neighbor_high_pm,0.133980
2,2MJ05565099$+$3006569,05565099$+$3006569,NaN,AGB,13.292,13_13.5,passed,0.142512
3,2MJ14313985$+$4149206,14313985$+$4149206,NaN,AGB,13.179,13_13.5,passed,0.176520


## 6. Sydney candidates not found in any pipeline output

In [22]:
if not sydney_reasons.empty and not sydney_valid.empty:
    # Which Sydney 2MASS (or ra/dec) appeared in at least one match?
    if "2MASS_clean" in sydney_reasons.columns:
        matched_2mass = set(sydney_reasons["2MASS_clean"].dropna().astype(str))
    else:
        matched_2mass = set()
    if "ra_deg" in sydney_reasons.columns and "dec_deg" in sydney_reasons.columns:
        matched_coords = set(zip(sydney_reasons["ra_deg"].round(6), sydney_reasons["dec_deg"].round(6)))
    else:
        matched_coords = set()
    not_in_pipeline = sydney_valid[
        ~sydney_valid["2MASS_clean"].astype(str).isin(matched_2mass)
        & ~sydney_valid.apply(lambda r: (round(r["ra_deg"], 6), round(r["dec_deg"], 6)) in matched_coords, axis=1)
    ]
    print(f"Sydney candidates with parseable coords: {len(sydney_valid)}")
    n_unique = sydney_reasons["2MASS_clean"].nunique() if "2MASS_clean" in sydney_reasons.columns else len(sydney_reasons)
    print(f"Unique Sydney candidates matched: {n_unique}")
    print(f"Sydney candidates NOT found in any pipeline output: {len(not_in_pipeline)}")
    if len(not_in_pipeline) > 0 and len(not_in_pipeline) <= 50:
        display(not_in_pipeline[[c for c in ["2MASS", "ASAS-SN ID", "Class", "ra_deg", "dec_deg"] if c in not_in_pipeline.columns]])
else:
    print("No Sydney or match data to compare.")

Sydney candidates with parseable coords: 416
Unique Sydney candidates matched: 4
Sydney candidates NOT found in any pipeline output: 412
